## cross validation for hyperparameter tuning using **GridSearchCV**

In [46]:
import pandas as pd
from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.metrics import accuracy_score,recall_score,precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

In [47]:
heart_df = pd.read_csv("/content/drive/MyDrive/Machine Learning/Supervised Learning/heart.csv")
heart_df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [48]:
X = heart_df.drop("target",axis=1)
y= heart_df['target']

In [49]:
print(heart_df.shape)
print(X.shape)
print(y.shape)

(303, 14)
(303, 13)
(303,)


In [50]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [51]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [52]:
classifier = KNeighborsClassifier()
param_grid = {"n_neighbors" : [3, 5, 7, 9]}

In [53]:
classifierCV = GridSearchCV(
    classifier,
    param_grid,
    cv=5,
    scoring="recall"
)

In [54]:
classifierCV.fit(X_train_scaled,y_train)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'n_neighbors': [3, 5, 7, 9]}, scoring='recall')

In [55]:
y_pred = classifierCV.predict(X_test_scaled)

In [56]:
print("Accuracy : ",accuracy_score(y_test,y_pred))
print("precision : ",precision_score(y_test,y_pred))
print("recall : ",recall_score(y_test,y_pred))

Accuracy :  0.9180327868852459
precision :  0.9354838709677419
recall :  0.90625


In [57]:
res = pd.DataFrame(classifierCV.cv_results_)
res

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_n_neighbors,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.002125,0.000418,0.005101,0.000263,3,{'n_neighbors': 3},0.851852,0.814815,0.962963,0.884615,0.807692,0.864387,0.056489,2
1,0.001630,0.000031,0.004910,0.000140,5,{'n_neighbors': 5},0.777778,0.814815,0.925926,0.923077,0.846154,0.857550,0.058803,3
2,0.001652,0.000069,0.004833,0.000165,7,{'n_neighbors': 7},0.814815,0.925926,0.925926,0.846154,0.846154,0.871795,0.045655,1
3,0.001576,0.000039,0.005012,0.000433,9,{'n_neighbors': 9},0.777778,0.888889,0.925926,0.846154,0.846154,0.856980,0.049556,4


In [58]:
print(classifierCV.best_params_)
print(classifierCV.best_score_)

{'n_neighbors': 7}
0.8717948717948717


## Mistake (data leakage) :

your workflow in gridsearchCV :
1. split data
2. sclated test data
3. use gridsearchCV find best parameter value
4. fit it and predict

-> In gridsearch use a scaled data , so validation data(in each fold) also scaled , means model have mean and standard deviation, so it is a data leakage

-> that why use pipeline for privant a data leakage

In [59]:
from sklearn.pipeline import Pipeline

In [60]:
pipeline = Pipeline([('scaler', StandardScaler()), ('knn_classifier',KNeighborsClassifier())])

In [61]:
param_grid = {"knn_classifier__n_neighbors" : [3, 5, 7, 9]}

classifierCV = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="recall"
)

classifierCV.fit(X_train,y_train) # pass normal data without scalling

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('knn_classifier',
                                        KNeighborsClassifier())]),
             param_grid={'knn_classifier__n_neighbors': [3, 5, 7, 9]},
             scoring='recall')

In [62]:
y_pred = classifierCV.predict(X_test)

In [63]:
print("Accuracy : ",accuracy_score(y_test,y_pred))
print("precision : ",precision_score(y_test,y_pred))
print("recall : ",recall_score(y_test,y_pred))

Accuracy :  0.9180327868852459
precision :  0.9354838709677419
recall :  0.90625


In [64]:
print(classifierCV.best_params_)
print(classifierCV.best_score_)

{'knn_classifier__n_neighbors': 7}
0.8717948717948717
